<a href="https://colab.research.google.com/github/Twinklingstar374/SectionB_G3_RetailAnalytics/blob/main/notebooks/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
local_raw = PROJECT_ROOT / 'data/raw/new_retail_data.csv'
colab_raw = Path('/content/new_retail_data.csv')

if local_raw.exists():
    RAW_PATH = local_raw
elif colab_raw.exists():
    RAW_PATH = colab_raw

PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

In [4]:
df = pd.read_csv(RAW_PATH)

print("Initial Shape:", df.shape)

Initial Shape: (302010, 30)


In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.isna().sum()

Transaction_ID      333
Customer_ID         308
Name                382
Email               347
Phone               362
Address             315
City                248
State               281
Zipcode             340
Country             271
Age                 173
Gender              317
Income              290
Customer_Segment    215
Date                359
Year                350
Month               273
Time                350
Total_Purchases     361
Amount              356
Total_Amount        350
Product_Category    283
Product_Brand       281
Product_Type          0
Feedback            184
Shipping_Method     337
Payment_Method      297
Order_Status        235
Ratings             184
products              0
dtype: int64

In [7]:
drop_cols = [
    'Name', 'Email', 'Phone', 'Address', 'Zipcode'
]
df.drop(columns=drop_cols, inplace=True)

In [8]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Total_Purchases'] = df['Total_Purchases'].fillna(0)
df['Amount'] = df['Amount'].fillna(df['Amount'].median())
df['Total_Amount'] = df['Total_Amount'].fillna(df['Total_Amount'].median())


In [9]:
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Customer_Segment'] = df['Customer_Segment'].fillna('Unknown')
df['Feedback'] = df['Feedback'].fillna('No Feedback')
df['Order_Status'] = df['Order_Status'].fillna('Pending')
df['Payment_Method'] = df['Payment_Method'].fillna('Unknown')
df['Shipping_Method'] = df['Shipping_Method'].fillna('Standard')
df['Income'] = df['Income'].fillna(df['Income'].mode()[0])
# Additional null handling for location and ID columns
df['City'] = df['City'].fillna(df['City'].mode()[0])
df['State'] = df['State'].fillna(df['State'].mode()[0])
df['Country'] = df['Country'].fillna(df['Country'].mode()[0])
df['Transaction_ID'] = df['Transaction_ID'].fillna(df['Transaction_ID'].median())
df['Customer_ID'] = df['Customer_ID'].fillna(df['Customer_ID'].median())
df['Year'] = df['Year'].fillna(df['Year'].mode()[0])
df['Month'] = df['Month'].fillna(df['Month'].mode()[0])
df['Product_Category'] = df['Product_Category'].fillna(df['Product_Category'].mode()[0])
df['Product_Brand'] = df['Product_Brand'].fillna(df['Product_Brand'].mode()[0])


In [10]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Date'] = df['Date'].ffill()

df['Time'] = df['Time'].fillna('00:00:00')


In [11]:
df['avg_order_value'] = df['Total_Amount'] / df['Total_Purchases']
df['avg_order_value'].replace([float('inf'), -float('inf')], 0, inplace=True)
df['avg_order_value'] = df['avg_order_value'].fillna(0)


/var/folders/v3/k8gbpw9s2x79mn7t4_knhyfm0000gn/T/ipykernel_37943/179207479.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['avg_order_value'].replace([float('inf'), -float('inf')], 0, inplace=True)


In [12]:
def age_group(age):
    if age < 25:
        return 'Young'
    elif age < 45:
        return 'Adult'
    else:
        return 'Senior'

df['age_group'] = df['Age'].apply(age_group)

In [13]:
def purchase_type(x):
    if x <= 5:
        return 'Low'
    elif x <= 15:
        return 'Medium'
    else:
        return 'High'

df['purchase_category'] = df['Total_Purchases'].apply(purchase_type)

In [14]:
def revenue_category(x):
    if x < 1000:
        return 'Low'
    elif x < 5000:
        return 'Medium'
    else:
        return 'High'

df['revenue_category'] = df['Total_Amount'].apply(revenue_category)

In [15]:
# Date already converted above — extract temporal features
df['day'] = df['Date'].dt.day
df['weekday'] = df['Date'].dt.day_name()
df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday'])


In [16]:
df['order_success'] = df['Order_Status'].apply(lambda x: 1 if x == 'Completed' else 0)

In [17]:
# Impute missing Ratings with median before categorisation
df['Ratings'] = df['Ratings'].fillna(df['Ratings'].median())

def rating_label(x):
    if x >= 4:
        return 'Good'
    elif x >= 2:
        return 'Average'
    else:
        return 'Poor'

df['rating_category'] = df['Ratings'].apply(rating_label)


In [18]:
def customer_value(row):
    if row['Total_Amount'] > 5000 and row['Total_Purchases'] > 10:
        return 'High Value'
    elif row['Total_Amount'] > 2000:
        return 'Medium Value'
    else:
        return 'Low Value'

df['customer_value_segment'] = df.apply(customer_value, axis=1)

In [19]:
df

,Transaction_ID,Customer_ID,City,State,Country,Age,Gender,Income,Customer_Segment,Date,...,avg_order_value,age_group,purchase_category,revenue_category,day,weekday,is_weekend,order_success,rating_category,customer_value_segment
0,8691788.0,37249.0,Dortmund,Berlin,Germany,21.0,Male,Low,Regular,2023-09-18,...,108.028757,Young,Low,Low,18,Monday,False,0,Good,Low Value
1,2174773.0,69749.0,Nottingham,England,UK,19.0,Female,Low,Premium,2023-12-31,...,403.353907,Young,Low,Low,31,Sunday,True,0,Good,Low Value
2,6679610.0,30192.0,Geelong,New South Wales,Australia,48.0,Male,Low,Regular,2023-04-26,...,354.477600,Senior,Low,Medium,26,Wednesday,False,0,Average,Low Value
3,7232460.0,62101.0,Edmonton,Ontario,Canada,56.0,Male,High,Premium,2023-05-08,...,352.407717,Senior,Medium,Medium,8,Monday,False,0,Good,Medium Value
4,4983775.0,27901.0,Bristol,England,UK,22.0,Male,Low,Premium,2024-01-10,...,124.276525,Young,Low,Low,10,Wednesday,False,0,Poor,Low Value
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302005,4246475.0,12104.0,Townsville,New South Wales,Australia,31.0,Male,Medium,Regular,2024-01-20,...,194.792597,Adult,Low,Low,20,Saturday,True,0,Poor,Low Value
302006,1197603.0,69772.0,Hanover,Berlin,Germany,35.0,Female,Low,New,2023-12-28,...,285.137301,Adult,Low,Low,28,Thursday,False,0,Good,Low Value
302007,7743242.0,28449.0,Brighton,England,UK,41.0,Male,Low,Premium,2024-02-27,...,60.701762,Adult,Low,Low,27,Tuesday,False,0,Average,Low Value
302008,9301950.0,45477.0,Halifax,Ontario,Canada,41.0,Male,Medium,New,2023-09-03,...,120.834784,Adult,Low,Low,3,Sunday,True,0,Good,Low Value


In [20]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to: {PROCESSED_PATH}')
print('Final shape:', df.shape)


Saved cleaned dataset to: /Users/bulbulagarwalla/SectionB_G3_RetailAnalytics/data/processed/cleaned_dataset.csv
Final shape: (302006, 35)
